# Sesión 3: Combinación de datos con Power Query## Códigos M - Ejemplos y EjerciciosEste notebook contiene el código M de la presentación **Combinación de datos con Power Query** para copiar y pegar directamente en el Editor de Power Query de Power BI.

---## SLIDE 6-9: Anexado de datos (Append Queries)### Concepto: Tipos de anexadoPower Query permite dos formas de anexar:1. **Anexado como nueva consulta**: Crea una nueva consulta resultado de la unión2. **Anexado dentro de una consulta existente**: Inserta registros en una consulta activa

### 1. Anexado simple: Table.Combine()Este es el método más directo para combinar múltiples tablas:

```mlet    // Cargar archivos individualmente    Enero = Excel.Workbook(        File.Contents("C:\\Datos\\enero.xlsx"),        null,        true    ){[Item="Sheet1", Kind="Sheet"]}[Data],        Febrero = Excel.Workbook(        File.Contents("C:\\Datos\\febrero.xlsx"),        null,        true    ){[Item="Sheet1", Kind="Sheet"]}[Data],        Marzo = Excel.Workbook(        File.Contents("C:\\Datos\\marzo.xlsx"),        null,        true    ){[Item="Sheet1", Kind="Sheet"]}[Data],        // Anexar (combinar) las tres tablas    SolicitudesTotales = Table.Combine({Enero, Febrero, Marzo})in    SolicitudesTotales```**Explicación:**- `Table.Combine()`: Agrega registros de múltiples tablas- `{Enero, Febrero, Marzo}`: Lista de tablas a combinar entre llaves- Power Query alinea columnas por nombre automáticamente- Columnas inexistentes en una tabla se rellenan con null

### 2. Anexado con preparación previa (normalización de encabezados)En la práctica, los archivos suelen tener nombres de columnas inconsistentes. Se deben normalizar ANTES del append:

```mlet    // Cargar archivos    Enero_Original = Excel.Workbook(        File.Contents("solicitudes_enero.xlsx"),        null,        true    ){0}[Data],        Febrero_Original = Excel.Workbook(        File.Contents("solicitudes_febrero.xlsx"),        null,        true    ){0}[Data],        Marzo_Original = Excel.Workbook(        File.Contents("solicitudes_marzo.xlsx"),        null,        true    ){0}[Data],        // ETAPA 1: Promocionar encabezados    Enero_Encabezados = Table.PromoteHeaders(Enero_Original),    Febrero_Encabezados = Table.PromoteHeaders(Febrero_Original),    Marzo_Encabezados = Table.PromoteHeaders(Marzo_Original),        // ETAPA 2: Normalizar nombres de columnas (eliminar espacios, trimmar)    Enero_Limpio = Table.TransformColumnNames(        Enero_Encabezados,        Text.Trim    ),    Febrero_Limpio = Table.TransformColumnNames(        Febrero_Encabezados,        Text.Trim    ),    Marzo_Limpio = Table.TransformColumnNames(        Marzo_Encabezados,        Text.Trim    ),        // ETAPA 3: Renombrar columnas para estandarizar (si nombres son diferentes)    Enero_Estandar = Table.RenameColumns(Enero_Limpio, {        {"Fecha_Solicitud", "Fecha"},        {"Tipo_Solicitud", "Tipo"},        {"Canal", "Canal_Ingreso"}    }),        Febrero_Estandar = Table.RenameColumns(Febrero_Limpio, {        {"FechaSolicitud", "Fecha"},        {"TipoSolicitud", "Tipo"},        {"CanalIngreso", "Canal_Ingreso"}    }),        Marzo_Estandar = Table.RenameColumns(Marzo_Limpio, {        {"Fecha", "Fecha"},        {"IdSolicitud", "ID_Solicitud"},        {"Tipo_Solicitud", "Tipo"},        {"Status", "Estado"},        {"Canal", "Canal_Ingreso"}    }),        // ETAPA 4: Reordenar columnas en todas las tablas (opcional pero recomendado)    Enero_Final = Table.ReorderColumns(Enero_Estandar,         {"ID_Solicitud", "Fecha", "Comuna", "Tipo", "Canal_Ingreso", "Estado"}),    Febrero_Final = Table.ReorderColumns(Febrero_Estandar,         {"ID_Solicitud", "Fecha", "Comuna", "Tipo", "Canal_Ingreso", "Estado"}),    Marzo_Final = Table.ReorderColumns(Marzo_Estandar,         {"ID_Solicitud", "Fecha", "Comuna", "Tipo", "Canal_Ingreso", "Estado"}),        // ETAPA 5: Agregar columna de origen (mes)    Enero_ConMes = Table.AddColumn(Enero_Final, "Mes", each "Enero", type text),    Febrero_ConMes = Table.AddColumn(Febrero_Final, "Mes", each "Febrero", type text),    Marzo_ConMes = Table.AddColumn(Marzo_Final, "Mes", each "Marzo", type text),        // ETAPA 6: Anexar (combinar)    SolicitudesTotales = Table.Combine({Enero_ConMes, Febrero_ConMes, Marzo_ConMes})in    SolicitudesTotales```**Explicación:**- Cada paso normaliza un aspecto diferente (encabezados, espacios, nombres)- `Table.RenameColumns()`: Cambia nombres de columnas- `Table.ReorderColumns()`: Organiza columnas en orden consistente- `Table.AddColumn()`: Agrega columna identificadora (Mes)- Finalmente, `Table.Combine()` une las tres tablas**Buena práctica:**- Estandarizar ANTES de anexar- Mantener trazabilidad agregando columna de origen

### 3. Validación post-anexadoDespués de anexar, verifica que los datos sean consistentes:

```mlet    SolicitudesTotales = // ... tabla anexada ...        // Agregar columna de control para verificar filas nulas    ConControl = Table.AddColumn(        SolicitudesTotales,        "Filas_Nulas",        each List.Count(            List.Select(                Record.FieldValues(_),                each _ = null            )        ),        type number    )in    ConControl```Luego en Power Query, puedes usar el perfil de columna para verificar:- Número de registros por Mes (debe sumar 106 = 34+35+37)- Valores únicos en Comuna- Distribución de Estados

---## SLIDE 12-18: Operaciones de Merge (Uniones entre tablas)### Concepto: Tipos de JOINUn merge combina dos tablas usando una columna clave en común. Los tipos de merge difieren en qué registros conservan:

### 1. LEFT OUTER JOIN (más común)Mantiene todos los registros de la tabla izquierda (principal) y agrega datos coincidentes de la derecha:

```mlet    // Tabla principal    SolicitudesTotales = // ... tabla de solicitudes ...        // Tabla de referencia    ComunasValidas = Csv.Document(        File.Contents("comunas_validas_sesion3.csv"),        [Delimiter=","]    ),    ComunasConEncabezados = Table.PromoteHeaders(ComunasValidas),        // Normalizar la columna de unión en ambas tablas    SolicitudesNormalizado = Table.TransformColumns(        SolicitudesTotales,        {"Comuna", Text.Trim, type text}    ),        ComunasNormalizado = Table.TransformColumns(        ComunasConEncabezados,        {"Nombre", Text.Trim, type text}    ),        // LEFT OUTER JOIN: mantener solicitudes aunque comuna no sea válida    MergeSolicitudes = Table.NestedJoin(        SolicitudesNormalizado,        {"Comuna"},        ComunasNormalizado,        {"Nombre"},        "DatosComuna",        JoinKind.LeftOuter    ),        // Expandir columnas de la tabla derecha    Expandido = Table.ExpandTableColumn(        MergeSolicitudes,        "DatosComuna",        {"Codigo_Comuna", "Region"},        {"Codigo_Comuna", "Region"}    ),        // Agregar columna indicadora: ¿es válida la comuna?    ConValidez = Table.AddColumn(        Expandido,        "Validez_Comuna",        each if [Codigo_Comuna] = null then "Inválida" else "Válida",        type text    )in    ConValidez```**Explicación:**- `Table.NestedJoin()`: Realiza la unión- Primera tabla: `SolicitudesNormalizado` (izquierda)- Segunda tabla: `ComunasNormalizado` (derecha)- Columnas de unión: `Comuna` y `Nombre`- `JoinKind.LeftOuter`: Mantiene todos los registros de la izquierda- `Table.ExpandTableColumn()`: Desanida los resultados para hacerlos visibles- `[Codigo_Comuna] = null`: Detecta registros sin coincidencia

### 2. INNER JOINSolo mantiene registros con coincidencia en ambas tablas:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...    ComunasValidas = // ... tabla de comunas ...        MergeSolicitudes = Table.NestedJoin(        SolicitudesTotales,        {"Comuna"},        ComunasValidas,        {"Nombre"},        "DatosComuna",        JoinKind.Inner  // Solo coincidencias    ),        Expandido = Table.ExpandTableColumn(        MergeSolicitudes,        "DatosComuna",        {"Codigo_Comuna"},        {"Codigo_Comuna"}    )in    Expandido```**Uso:** Cuando solo interesa trabajar con datos válidos (sin comunas inválidas)

### 3. FULL OUTER JOINMantiene TODOS los registros de ambas tablas (coincidentes y no coincidentes):

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...    ComunasValidas = // ... tabla de comunas ...        MergeSolicitudes = Table.NestedJoin(        SolicitudesTotales,        {"Comuna"},        ComunasValidas,        {"Nombre"},        "DatosComuna",        JoinKind.FullOuter  // Todos de ambas tablas    ),        Expandido = Table.ExpandTableColumn(        MergeSolicitudes,        "DatosComuna",        {"Codigo_Comuna"},        {"Codigo_Comuna"}    )in    Expandido```**Uso:** Diagnóstico de cobertura (qué comunas no tienen solicitudes, cuáles son inválidas)

### 4. LEFT ANTI JOINSolo registros de la IZQUIERDA que NO tienen coincidencia en la derecha:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...    ComunasValidas = // ... tabla de comunas ...        MergeSolicitudes = Table.NestedJoin(        SolicitudesTotales,        {"Comuna"},        ComunasValidas,        {"Nombre"},        "DatosComuna",        JoinKind.LeftAnti  // Sin coincidencia    )in    MergeSolicitudes```**Uso:** Encontrar comunas inválidas o errores de tipeo

### 5. RIGHT ANTI JOINSolo registros de la DERECHA que NO tienen coincidencia en la izquierda:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...    ComunasValidas = // ... tabla de comunas ...        MergeSolicitudes = Table.NestedJoin(        SolicitudesTotales,        {"Comuna"},        ComunasValidas,        {"Nombre"},        "DatosComuna",        JoinKind.RightAnti  // Comunas válidas sin solicitudes    )in    MergeSolicitudes```**Uso:** Validar qué comunas válidas no han tenido solicitudes

### Buenas prácticas en Merge1. **Normalizar encabezados antes:**

```m// Aplicar a ambas tablasSolicitudesPrep = Table.TransformColumns(    SolicitudesTotales,    {"Comuna", each Text.Upper(Text.Trim(_)), type text}),ComunasPrep = Table.TransformColumns(    ComunasValidas,    {"Nombre", each Text.Upper(Text.Trim(_)), type text})```2. **Validar unicidad de clave:**- Si la clave no es única en la tabla derecha, se duplicarán registros- Usar LEFT ANTI o RIGHT ANTI para diagnosticar3. **Expandir correctamente:**

```m// Expandir solo columnas necesarias, no todasTable.ExpandTableColumn(    MergeSolicitudes,    "DatosComuna",    {"Codigo_Comuna", "Region"},  // Solo estos campos    {"Codigo_Comuna", "Region"})```

---## SLIDE 20-24: Agrupamiento (Group By)### 1. Agrupamiento simple: Contar por grupo

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...        // Agrupar por Canal y contar registros    PorCanal = Table.Group(        SolicitudesTotales,        {"Canal_Ingreso"},        {            {"Total_Solicitudes", Table.RowCount, Int64.Type}        }    )in    PorCanal```**Resultado:**| Canal_Ingreso | Total_Solicitudes ||---------------|-------------------|| Ventanilla    | 25                || WEB           | 28                || Call center   | 30                |

### 2. Agrupamiento por múltiples columnasAgrupa por dos o más campos categóricos:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...        // Agrupar por Comuna y Canal (combinación)    PorComunaYCanal = Table.Group(        SolicitudesTotales,        {"Comuna", "Canal_Ingreso"},        {            {"Total_Solicitudes", Table.RowCount, Int64.Type}        }    )in    PorComunaYCanal```**Resultado:**| Comuna     | Canal_Ingreso | Total_Solicitudes ||------------|---------------|-------------------|| Santiago   | Ventanilla    | 8                 || Santiago   | WEB           | 12                || Providencia| Call center   | 5                 |

### 3. Múltiples agregaciones simultáneasCalcula varias métricas en un solo Group By:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...        // Convertir Fecha a tipo date si es necesario    ConTipo = Table.TransformColumnTypes(        SolicitudesTotales,        {"Fecha", type date}    ),        // Agrupar por Comuna con múltiples cálculos    ResumenPorComuna = Table.Group(        ConTipo,        {"Comuna"},        {            {"Total_Solicitudes", Table.RowCount, Int64.Type},            {"Fecha_Mas_Reciente", each List.Max([Fecha]), type date},            {"Fecha_Mas_Antigua", each List.Min([Fecha]), type date},            {"Tipos_Unicos", each List.Count(List.Distinct([Tipo])), Int64.Type}        }    )in    ResumenPorComuna```**Resultado:**| Comuna     | Total_Solicitudes | Fecha_Mas_Reciente | Fecha_Mas_Antigua | Tipos_Unicos ||------------|-------------------|-------------------|-------------------|--------------|| Santiago   | 20                | 2024-03-31        | 2024-01-01        | 3            |

### 4. Agrupamiento anidadoConserva los datos originales dentro de cada grupo (tabla anidada):

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...        // Agrupar pero mantener subtabla con detalle    GruposPorCanal = Table.Group(        SolicitudesTotales,        {"Canal_Ingreso"},        {            {"Total", Table.RowCount, Int64.Type},            {"Detalle", each _, type table}  // Mantiene todos los registros        }    )in    GruposPorCanal```**Ventaja:** Permite aplicar filtros o transformaciones adicionales dentro de cada grupo

---## SLIDE 27-32: Pivoteo y Despivoteo### 1. Pivoteo: Convertir filas a columnasÚtil para crear reportes con meses o períodos como columnas:

```mlet    // Tabla de entrada (formato largo):    // | Comuna     | Mes      | Total |    // | Santiago   | Enero    | 10    |    // | Santiago   | Febrero  | 15    |    // | Providencia| Enero    | 8     |        TablaLarga = // ... tabla de solicitudes agrupada ...        // PIVOTEAR: convertir Mes en columnas    TablaPivotea = Table.Pivot(        TablaLarga,        List.Distinct(TablaLarga[Mes]),  // Valores únicos de Mes        "Mes",                            // Columna a pivotar        "Total",                          // Columna de valores        List.Sum                          // Función de agregación    )in    TablaPivotea    // Resultado (formato ancho):    // | Comuna      | Enero | Febrero | Marzo |    // | Santiago    | 10    | 15      | 12    |    // | Providencia | 8     | 10      | 9     |```**Explicación:**- `List.Distinct()`: Obtiene valores únicos (Enero, Febrero, Marzo)- `"Mes"`: La columna cuyos valores se convierten en encabezados- `"Total"`: La columna con valores a pivotar- `List.Sum`: Si hay múltiples valores por combinación, súmalos

### 2. Pivoteo paso a paso (con preparación)En la práctica, es mejor preparar los datos antes de pivotar:

```mlet    SolicitudesTotales = // ... tabla de solicitudes ...        // PASO 1: Filtrar datos válidos    SoloValidas = Table.SelectRows(        SolicitudesTotales,        each [Validez_Comuna] = "Válida"    ),        // PASO 2: Agrupar por Comuna, Tipo, Mes    Agrupado = Table.Group(        SoloValidas,        {"Comuna", "Tipo", "Mes"},        {            {"Total", Table.RowCount, Int64.Type}        }    ),        // PASO 3: Extraer valores únicos de Mes    MesesUnicos = List.Distinct(Agrupado[Mes]),        // PASO 4: Pivotar    Pivoteado = Table.Pivot(        Agrupado,        MesesUnicos,        "Mes",        "Total",        List.Sum    ),        // PASO 5: Reemplazar nulos con 0 (opcional)    SinNulos = Table.ReplaceValue(        Pivoteado,        null,        0,        Replacer.ReplaceValue,        MesesUnicos    )in    SinNulos```

### 3. Despivoteo: Convertir columnas a filasÚtil para normalizar datos que vienen en formato ancho:

```mlet    // Tabla de entrada (formato ancho):    // | Comuna      | Enero | Febrero | Marzo |    // | Santiago    | 10    | 15      | 12    |        TablaAncha = // ... tabla con meses como columnas ...        // DESPIVOTEAR: convertir Enero, Febrero, Marzo en filas    TablaLarga = Table.UnpivotOtherColumns(        TablaAncha,        {"Comuna"},        // Columnas que NO se transforman (clave)        "Mes",            // Nombre de la columna de atributo        "Total"           // Nombre de la columna de valor    )in    TablaLarga    // Resultado (formato largo):    // | Comuna      | Mes      | Total |    // | Santiago    | Enero    | 10    |    // | Santiago    | Febrero  | 15    |    // | Santiago    | Marzo    | 12    |```**Explicación:**- `{"Comuna"}`: Columnas que no se despivotean (clave)- Todas las otras columnas (Enero, Febrero, Marzo) se despivotean- Crea dos columnas nuevas: `Mes` (atributo) y `Total` (valor)

### 4. Despivoteo específico: Seleccionar columnasSi solo quieres despivotear algunas columnas:

```mlet    TablaAncha = // ... tabla ...        // DESPIVOTEAR solo Enero, Febrero, Marzo    TablaLarga = Table.Unpivot(        TablaAncha,        {"Enero", "Febrero", "Marzo"},  // Columnas a despivotear        "Mes",                          // Nombre de atributo        "Total"                         // Nombre de valor    )in    TablaLarga```

---## Resumen de funciones clave### Append/Combine- `Table.Combine()`: Agrega registros de múltiples tablas### Merge/Join- `Table.NestedJoin()`: Uniona dos tablas por columna clave- `Table.ExpandTableColumn()`: Desanida resultados del join- `JoinKind.LeftOuter`, `JoinKind.Inner`, `JoinKind.FullOuter`, `JoinKind.LeftAnti`, `JoinKind.RightAnti`### Group By- `Table.Group()`: Agrupa por columna(s) y aplica funciones### Transformación de estructura- `Table.Pivot()`: Convierte filas a columnas- `Table.Unpivot()`: Convierte columnas específicas a filas- `Table.UnpivotOtherColumns()`: Convierte todas EXCEPTO clave a filas### Preparación- `Table.TransformColumnNames()`: Normaliza nombres (trim, lower, etc)- `Table.RenameColumns()`: Cambia nombres de columnas- `Table.ReorderColumns()`: Reordena columnas- `Table.AddColumn()`: Agrega nueva columna con valores derivados

---## Notas finales- **Normaliza ANTES de combinar:** Names, espacios, tipos de dato- **Documenta cada paso:** Especialmente merges y grupos complejos- **Valida después:** Usa perfil de columna, verifica conteos- **Evita many-to-many merge:** Asegura que claves sean únicas- **Orden de operaciones:** Append → Merge → Group → Pivot/Unpivot- Copiar/pegar directamente en Power Query Editor- Reemplaza rutas de archivo y nombres de columnas según tu caso